# 00. Colab 환경 설정 및 MVTec 전체 category 확인

이 노트북은 Google Drive의 `data/raw/mvtec` 아래 모든 MVTec AD category를 탐색하고, 학습 대상 manifest를 만듭니다.

- `pip install -r requirements.txt`는 실행하지 않습니다.
- Drive의 `raw/mvtec` 폴더에 있는 모든 category를 대상으로 합니다.
- 결과 manifest는 Drive와 GitHub repo의 `docs/assets/results`에 저장합니다.

In [1]:
# ===== 공통 환경 설정: Colab + Drive + GitHub repo + 안전한 import 경로 =====
# 이 셀은 모든 노트북에서 가장 먼저 실행하세요.
# 핵심 원칙:
# - requirements.txt 전체 설치 금지
# - numpy / pandas / torch / torchvision / opencv / scikit-learn 강제 재설치 금지
# - 데이터, checkpoint, deploy bundle은 Google Drive에 저장
# - GitHub에는 코드, README, 작은 시각화 파일, demo sample 이미지만 업로드

import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
from getpass import getpass

GITHUB_REPO_URL = "https://github.com/wnstjq0915/DefectVision-AD-Proj.git"
GITHUB_USERNAME = "wnstjq0915"
GITHUB_EMAIL = "wnstjq0915@gmail.com"
PROJECT_DIR = Path("/content/DefectVision-AD-Proj")
PROJECT_NAME = "DefectVision-AD"

try:
    from google.colab import drive
    IN_COLAB = True
    drive.mount('/content/drive')
except Exception as exc:
    IN_COLAB = False
    print('Colab이 아닌 환경입니다. Drive mount 생략:', repr(exc))

DRIVE_ROOT = Path('/content/drive/MyDrive') / PROJECT_NAME if IN_COLAB else Path.cwd() / 'drive_sim' / PROJECT_NAME
DATA_ROOT = DRIVE_ROOT / 'data' / 'raw'
MVTEC_ROOT = DATA_ROOT / 'mvtec'
VISA_ROOT = DATA_ROOT / 'visa_mvtec'
OUTPUT_ROOT = DRIVE_ROOT / 'outputs'
RESULT_ROOT = OUTPUT_ROOT / 'multi_category_results'
CHECKPOINT_ROOT = OUTPUT_ROOT / 'checkpoints'
DEPLOY_ROOT = DRIVE_ROOT / 'deploy'

for p in [DRIVE_ROOT, DATA_ROOT, OUTPUT_ROOT, RESULT_ROOT, CHECKPOINT_ROOT, DEPLOY_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# repo clone 또는 재사용
FORCE_RECLONE = False
if FORCE_RECLONE and PROJECT_DIR.exists():
    shutil.rmtree(PROJECT_DIR)

if not PROJECT_DIR.exists():
    print('GitHub repo clone:', GITHUB_REPO_URL)
    subprocess.check_call(['git', 'clone', GITHUB_REPO_URL, str(PROJECT_DIR)])
else:
    print('기존 GitHub repo 사용:', PROJECT_DIR)

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

# Git 사용자 정보는 commit용. push 인증은 별도 token 환경변수 사용.
subprocess.run(['git', 'config', '--global', 'user.email', GITHUB_EMAIL], check=False)
subprocess.run(['git', 'config', '--global', 'user.name', GITHUB_USERNAME], check=False)

print('IN_COLAB       =', IN_COLAB)
print('PROJECT_DIR    =', PROJECT_DIR)
print('DRIVE_ROOT     =', DRIVE_ROOT)
print('MVTEC_ROOT     =', MVTEC_ROOT)
print('VISA_ROOT      =', VISA_ROOT)
print('RESULT_ROOT    =', RESULT_ROOT)
print('CHECKPOINT_ROOT=', CHECKPOINT_ROOT)

# Colab 기본 패키지 버전 확인. 버전 꼬임 방지를 위해 강제 설치하지 않음.
import importlib
print('\n===== 주요 패키지 버전 =====')
for name in ['numpy', 'pandas', 'torch', 'torchvision', 'cv2', 'sklearn', 'matplotlib', 'PIL', 'yaml', 'tqdm']:
    try:
        mod = importlib.import_module(name)
        ver = getattr(mod, '__version__', 'unknown')
        if name == 'PIL':
            from PIL import Image
            ver = Image.__version__
        print(f'{name:12s}: {ver}')
    except Exception as exc:
        print(f'{name:12s}: IMPORT ERROR -> {repr(exc)}')

# 작은 유틸 패키지만 누락 시 설치. 핵심 ML 패키지는 설치하지 않음.
for import_name, pip_name in [('yaml', 'PyYAML'), ('tqdm', 'tqdm')]:
    try:
        importlib.import_module(import_name)
    except Exception:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])

# torch.load weights_only 기본값 변경에 대비한 안전 패치
# PyTorch 2.6+에서는 torch.load 기본 weights_only가 바뀌어 기존 checkpoint 로드가 실패할 수 있음.
inference_path = PROJECT_DIR / 'src' / 'inference.py'
if inference_path.exists():
    text = inference_path.read_text(encoding='utf-8')
    old = 'checkpoint = torch.load(checkpoint_path, map_location=resolved_device)'
    new = """\n    try:\n        checkpoint = torch.load(checkpoint_path, map_location=resolved_device, weights_only=False)\n    except TypeError:\n        checkpoint = torch.load(checkpoint_path, map_location=resolved_device)\n    """.rstrip()
    if old in text:
        text = text.replace(old, new)
        inference_path.write_text(text, encoding='utf-8')
        print('patched:', inference_path)

print('\n현재 작업 디렉토리:', Path.cwd())
print('src 존재 여부:', (PROJECT_DIR / 'src').exists())

Mounted at /content/drive
GitHub repo clone: https://github.com/wnstjq0915/DefectVision-AD-Proj.git
IN_COLAB       = True
PROJECT_DIR    = /content/DefectVision-AD-Proj
DRIVE_ROOT     = /content/drive/MyDrive/DefectVision-AD
MVTEC_ROOT     = /content/drive/MyDrive/DefectVision-AD/data/raw/mvtec
VISA_ROOT      = /content/drive/MyDrive/DefectVision-AD/data/raw/visa_mvtec
RESULT_ROOT    = /content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results
CHECKPOINT_ROOT= /content/drive/MyDrive/DefectVision-AD/outputs/checkpoints

===== 주요 패키지 버전 =====
numpy       : 2.0.2
pandas      : 2.2.2
torch       : 2.11.0+cpu
torchvision : 0.26.0+cpu
cv2         : 4.13.0
sklearn     : 1.6.1
matplotlib  : 3.10.0
PIL         : 11.3.0
yaml        : 6.0.3
tqdm        : 4.67.3
patched: /content/DefectVision-AD-Proj/src/inference.py

현재 작업 디렉토리: /content/DefectVision-AD-Proj
src 존재 여부: True


In [2]:
# ===== Dataset discovery / manifest helper =====
from pathlib import Path
import csv
import json
import random
from collections import Counter, defaultdict
from typing import Iterable

IMAGE_EXTENSIONS = {'.bmp', '.jpg', '.jpeg', '.png', '.tif', '.tiff'}


def image_files(directory: Path):
    if not directory.exists():
        return []
    return sorted(p for p in directory.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)


def discover_mvtec_categories(root: Path):
    root = Path(root)
    if not root.exists():
        return []
    cats = []
    for p in sorted(root.iterdir()):
        if not p.is_dir() or p.name.startswith('.'):
            continue
        if (p / 'train' / 'good').exists() and (p / 'test').exists():
            cats.append(p.name)
    return cats


def summarize_mvtec_category(root: Path, category: str):
    cat_root = Path(root) / category
    row = {
        'dataset': 'mvtec',
        'category': category,
        'category_root': str(cat_root),
        'train_good': 0,
        'test_good': 0,
        'test_anomaly': 0,
        'test_total': 0,
        'defect_types': '',
        'has_ground_truth': False,
        'status': 'missing',
    }
    if not cat_root.exists():
        return row
    train_good = image_files(cat_root / 'train' / 'good')
    test_counts = {}
    test_root = cat_root / 'test'
    if test_root.exists():
        for d in sorted(x for x in test_root.iterdir() if x.is_dir()):
            test_counts[d.name] = len(image_files(d))
    row['train_good'] = len(train_good)
    row['test_good'] = test_counts.get('good', 0)
    row['test_anomaly'] = sum(v for k, v in test_counts.items() if k != 'good')
    row['test_total'] = sum(test_counts.values())
    row['defect_types'] = ', '.join(k for k in sorted(test_counts) if k != 'good')
    row['has_ground_truth'] = (cat_root / 'ground_truth').exists()
    row['status'] = 'ok' if row['train_good'] and row['test_total'] else 'incomplete'
    return row


def write_csv(rows, path: Path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text('', encoding='utf-8')
        return path
    fieldnames = list(rows[0].keys())
    with path.open('w', encoding='utf-8', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)
    return path

In [3]:
# ===== MVTec 전체 category 탐색 =====
categories = discover_mvtec_categories(MVTEC_ROOT)
print('MVTec root:', MVTEC_ROOT)
print('발견 category 수:', len(categories))
for c in categories:
    print(' -', c)

if not categories:
    raise FileNotFoundError(
        f'MVTec category를 찾지 못했습니다: {MVTEC_ROOT}\n'
        'Drive에 /content/drive/MyDrive/DefectVision-AD/data/raw/mvtec/<category>/train/good 구조가 있어야 합니다.'
    )

summary_rows = [summarize_mvtec_category(MVTEC_ROOT, c) for c in categories]

manifest_drive = RESULT_ROOT / 'mvtec_category_manifest.csv'
manifest_repo = PROJECT_DIR / 'docs' / 'assets' / 'results' / 'mvtec_category_manifest.csv'
write_csv(summary_rows, manifest_drive)
write_csv(summary_rows, manifest_repo)

print('saved:', manifest_drive)
print('saved:', manifest_repo)

try:
    import pandas as pd
    from IPython.display import display
    display(pd.DataFrame(summary_rows))
except Exception as exc:
    print('pandas 표시 실패. CSV를 확인하세요:', repr(exc))
    for row in summary_rows:
        print(row)

MVTec root: /content/drive/MyDrive/DefectVision-AD/data/raw/mvtec
발견 category 수: 15
 - bottle
 - cable
 - capsule
 - carpet
 - grid
 - hazelnut
 - leather
 - metal_nut
 - pill
 - screw
 - tile
 - toothbrush
 - transistor
 - wood
 - zipper
saved: /content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results/mvtec_category_manifest.csv
saved: /content/DefectVision-AD-Proj/docs/assets/results/mvtec_category_manifest.csv


,dataset,category,category_root,train_good,test_good,test_anomaly,test_total,defect_types,has_ground_truth,status
0,mvtec,bottle,/content/drive/MyDrive/DefectVision-AD/data/ra...,209,20,63,83,"broken_large, broken_small, contamination",True,ok
1,mvtec,cable,/content/drive/MyDrive/DefectVision-AD/data/ra...,224,58,92,150,"bent_wire, cable_swap, combined, cut_inner_ins...",True,ok
2,mvtec,capsule,/content/drive/MyDrive/DefectVision-AD/data/ra...,219,23,109,132,"crack, faulty_imprint, poke, scratch, squeeze",True,ok
3,mvtec,carpet,/content/drive/MyDrive/DefectVision-AD/data/ra...,280,28,89,117,"color, cut, hole, metal_contamination, thread",True,ok
4,mvtec,grid,/content/drive/MyDrive/DefectVision-AD/data/ra...,264,21,57,78,"bent, broken, glue, metal_contamination, thread",True,ok
5,mvtec,hazelnut,/content/drive/MyDrive/DefectVision-AD/data/ra...,391,40,70,110,"crack, cut, hole, print",True,ok
6,mvtec,leather,/content/drive/MyDrive/DefectVision-AD/data/ra...,245,32,92,124,"color, cut, fold, glue, poke",True,ok
7,mvtec,metal_nut,/content/drive/MyDrive/DefectVision-AD/data/ra...,220,22,93,115,"bent, color, flip, scratch",True,ok
8,mvtec,pill,/content/drive/MyDrive/DefectVision-AD/data/ra...,267,26,141,167,"color, combined, contamination, crack, faulty_...",True,ok
9,mvtec,screw,/content/drive/MyDrive/DefectVision-AD/data/ra...,320,41,119,160,"manipulated_front, scratch_head, scratch_neck,...",True,ok


In [4]:
# ===== 학습 대상 category 설정 파일 생성 =====
# 전체 실행 시 15개 category 전체가 들어갑니다.
# 일부 category만 먼저 돌리고 싶으면 아래 include_categories 값을 수정하세요.

include_categories = categories
# 예: include_categories = ['bottle', 'cable', 'wood']

run_plan = {
    'dataset': 'mvtec',
    'root': str(MVTEC_ROOT),
    'categories': include_categories,
    'notes': 'MVTec AD categories discovered from Google Drive. Train uses train/good only; test samples are not training images.',
}

plan_path = RESULT_ROOT / 'run_plan_mvtec_all_categories.json'
plan_path.write_text(json.dumps(run_plan, indent=2, ensure_ascii=False), encoding='utf-8')
print('saved:', plan_path)
print(json.dumps(run_plan, indent=2, ensure_ascii=False))

saved: /content/drive/MyDrive/DefectVision-AD/outputs/multi_category_results/run_plan_mvtec_all_categories.json
{
  "dataset": "mvtec",
  "root": "/content/drive/MyDrive/DefectVision-AD/data/raw/mvtec",
  "categories": [
    "bottle",
    "cable",
    "capsule",
    "carpet",
    "grid",
    "hazelnut",
    "leather",
    "metal_nut",
    "pill",
    "screw",
    "tile",
    "toothbrush",
    "transistor",
    "wood",
    "zipper"
  ],
  "notes": "MVTec AD categories discovered from Google Drive. Train uses train/good only; test samples are not training images."
}


In [5]:
# ===== repo에 결과 디렉토리와 .gitignore 정리 =====
assets_dir = PROJECT_DIR / 'docs' / 'assets' / 'results'
assets_dir.mkdir(parents=True, exist_ok=True)

# 모델 checkpoint와 대용량 결과는 git에 올리지 않도록 보강
ignore_path = PROJECT_DIR / '.gitignore'
existing = ignore_path.read_text(encoding='utf-8') if ignore_path.exists() else ''
extra = """
# DefectVision large artifacts
data/raw/
data/processed/
outputs/checkpoints/
outputs/heatmaps/
outputs/metrics/*.json
*.pt
*.pth
*.ckpt
*.onnx
*.zip
.DS_Store
"""
if 'DefectVision large artifacts' not in existing:
    ignore_path.write_text(existing.rstrip() + '\n' + extra + '\n', encoding='utf-8')
    print('updated:', ignore_path)
else:
    print('.gitignore already contains DefectVision rules')

updated: /content/DefectVision-AD-Proj/.gitignore
